# enwiktionary HTML dump

Provided as part of the [Wikimedia Enterprise HTML Dumps](https://dumps.wikimedia.org/other/enterprise_html/). Specifically here: https://dumps.wikimedia.org/other/enterprise_html/runs/. There will be files like the following:

```
enwiktionary-NS0-20250201-ENTERPRISE-HTML.json...> 01-Feb-2025 11:43         15443888277
enwiktionary-NS0-20250201-ENTERPRISE-STATS.json    01-Feb-2025 11:43                 121
enwiktionary-NS10-20250201-ENTERPRISE-HTML.json..> 02-Feb-2025 00:09           168297736
enwiktionary-NS10-20250201-ENTERPRISE-STATS.json   02-Feb-2025 00:09                 121
enwiktionary-NS14-20250201-ENTERPRISE-HTML.json..> 01-Feb-2025 19:05           576717474
enwiktionary-NS14-20250201-ENTERPRISE-STATS.json   01-Feb-2025 19:05                 121
enwiktionary-NS6-20250201-ENTERPRISE-HTML.json...> 02-Feb-2025 04:53               14286
enwiktionary-NS6-20250201-ENTERPRISE-STATS.json    02-Feb-2025 04:53                 121
```

The `NS` in the filenames stands for the MediaWiki namespace:

- NS0: Main namespace (actual dictionary entries)
- NS6: File namespace (used for media like images)
- NS10: Template namespace (for reusable page components)
- NS14: Category namespace (for organizing pages into categories)

We want the one like `enwiktionary-NS0-20250201-ENTERPRISE-HTML.json` which contains the actual dictionary entries.

The file is quite large, 14.4 GB on 2025-02-01. The download is slow and sometimes fails partway through.

Each dump output file consists of a tar.gz archive which, when uncompressed and untarred, contains one file, with a single line per article, in json format. Among the attributes defined in the file are the following below, however to see all attributes please visit the [data dictionary for Wikimedia Enterprise APIs](https://enterprise.wikimedia.com/docs/data-dictionary/):

- `name`: the title of the article
- `identifier`: the page id
- `date modified`: the last time the page was modified
- `version`: a compound structure representing the page revision
- `url`: the full url to the page on the wiki
- `namespace`: a compound structure representing the namespace of the page
- `in_language`: a compound structure representing the language of the wiki that has the page
- `article_body`: has two attributes, `html` and `wikitext`, which contain the html and wikitext of the page, respectively
- `wikitext`: the plain wikitext of the page without additional markup
- `license`: license information for the page

Check back on beta structured contents https://enterprise.wikimedia.com/docs/on-demand/#article-structured-contents-beta -- this is offered through API currently but not available in the dumps yet.


In [ ]:
!mkdir -p enwiktionary-NS0-20250201-ENTERPRISE-HTML-extracted
!pigz -dc enwiktionary-NS0-20250201-ENTERPRISE-HTML.json.tar.gz | tar xvf - -C enwiktionary-NS0-20250201-ENTERPRISE-HTML-extracted

In [ ]:
from pathlib import Path
import json

n66 = Path(
    "enwiktionary-NS0-20250201-ENTERPRISE-HTML-extracted/enwiktionary_namespace_0_66.ndjson"
)

# read lines from the file
with n66.open() as f:
    for line in f.readlines():
        content = json.loads(line)
        if content.get("name") == "test":
            article_body = content.pop("article_body")
            with open("test.html", "w") as f:
                f.write(article_body["html"])
            with open("test.wikitext", "w") as f:
                f.write(article_body["wikitext"])
            with open("test.json", "w") as f:
                f.write(json.dumps(content, indent=2, ensure_ascii=False))

In [ ]:
from lxml import etree
import os


def write_chunk(output_dir, chunk, chunk_index, current_size):
    with open(os.path.join(output_dir, f"chunk_{chunk_index}.xml"), "w") as f:
        f.write("<root>\n")
        f.writelines(chunk)
        f.write("</root>\n")
    print(f"Created chunk_{chunk_index}.xml with size {current_size} bytes")


def split_xml(file_path, output_dir, tag, chunk_size, namespace):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    ns_tag = f"{{{namespace}}}{tag}"
    context = etree.iterparse(file_path, events=("end",), tag=ns_tag)
    chunk = []
    chunk_index = 0
    current_size = 0

    for event, elem in context:
        chunk.append(etree.tostring(elem, pretty_print=True, encoding="unicode"))
        current_size += len(chunk[-1])

        if current_size >= chunk_size:
            write_chunk(output_dir, chunk, chunk_index, current_size)
            chunk = []
            current_size = 0
            chunk_index += 1

        elem.clear()
        while elem.getprevious() is not None:
            del elem.getparent()[0]

    if chunk:
        write_chunk(output_dir, chunk, chunk_index, current_size)


# Example usage
split_xml(
    "test-pages-articles.xml", "output_chunks", "page", 1 * 1024 * 1024, namespace
)  # 1 MB chunks

# refs

- https://github.com/tylertitsworth/multi-mediawiki-rag/blob/main/embed.py
- https://huggingface.co/learn/cookbook/en/structured_generation

# preprocesing

- preprocess html to get list of all unique (lang, term, href, title) tuples


In [ ]:
import pandas as pd
import concurrent.futures
from bs4 import BeautifulSoup
import os
import json
from pathlib import Path


def find_term_links(html_content):
    soup = BeautifulSoup(html_content, "lxml")
    term_links = set()

    for i_element in soup.find_all("i", lang=True):
        lang = i_element.get("lang", "")
        for a_element in i_element.find_all("a"):
            term = a_element.get_text(strip=True)
            href = a_element.get("href", "")
            title = a_element.get("title", "")
            term_links.add((lang, term, href, title))

    return term_links


def process_ndjson_file(ndjson_file_path):
    term_links = set()

    with open(ndjson_file_path, "r") as f:
        for line in f:
            content = json.loads(line)
            article_body = content.get("article_body", {})
            html_content = article_body.get("html", "")
            if html_content:
                term_links.update(find_term_links(html_content))

    return term_links


def process_all_files(file_paths):
    all_term_links = set()

    with concurrent.futures.ThreadPoolExecutor() as executor:
        futures = (
            executor.submit(process_ndjson_file, file_path) for file_path in file_paths
        )
        for future in concurrent.futures.as_completed(futures):
            term_links = future.result()
            all_term_links.update(term_links)

    return all_term_links


# Example usage
file_paths = Path("enwiktionary-NS0-20250201-ENTERPRISE-HTML-extracted").glob(
    "*.ndjson"
)
all_term_links = process_all_files(file_paths)

# Write to CSV, sort by lang and term
df = pd.DataFrame(all_term_links, columns=["lang", "term", "href", "title"])
df.sort_values(["lang", "term"], inplace=True)
df.to_csv("term_links.csv", index=False)

In [ ]:
import pandas as pd

# Read the CSV file
df = pd.read_csv("term_links.csv")

# Group by 'lang' and 'term' and filter groups with more than one unique 'href' or 'title'
duplicate_groups = df.groupby(["lang", "term"]).filter(
    lambda x: x["href"].nunique() > 1
)

# Display the duplicate groups
print(duplicate_groups)

In [ ]:
from bs4 import BeautifulSoup
import re


def render_html_to_text(html_content):
    soup = BeautifulSoup(html_content, "html.parser")
    text = soup.get_text()
    # Remove excessive whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text


text_content = render_html_to_text(html_content)
print(text_content)

In [ ]:
from typing import List, Optional
from pydantic import BaseModel


class Item(BaseModel):
    id: int
    etyNum: int
    lang: str
    term: str
    imputed: bool
    reconstructed: bool
    url: Optional[str] = None
    pos: Optional[List[str]] = None
    gloss: Optional[List[str]] = None
    romanization: Optional[str] = None


class Etymology(BaseModel):
    item: Item
    etyMode: Optional[str] = None
    etyOrder: int
    parents: List["Etymology"] = []


ety_schema = Etymology.model_json_schema()

In [ ]:
prompt_template = """
You are an expert in word etymologies. Given a word's etymology, your task is to convert it to JSON format. Here is the word and its etymology:

{term} ({lang} {pos})

{ety}
"""

In [ ]:
prompt = prompt_template.format(
    term="hap",
    lang="English",
    pos="noun",
    ety="From Middle English hap, happe (“chance, hap, luck, fortune”), potentially cognate with or from Old English ġehæp (“fit, convenient”) and/or Old Norse happ (“hap, chance, good luck”), from Proto-Germanic *hampą (“convenience, happiness”), from Proto-Indo-European *kob- (“good fortune, prophecy; to bend, bow, fit in, work, succeed”).",
)

In [ ]:
# model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
# model_name = "Qwen/Qwen2.5-7B-Instruct-GPTQ-Int8"
model_name = "Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4"

# !vllm serve $model_name
# !vllm --version

# in separate terminal, run `vllm serve Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4 --max-model-len 4096`

from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="-",
)

completion = client.chat.completions.create(
    model=model_name,
    messages=[
        {
            "role": "user",
            # "content": prompt,
            "content": "hi",
        }
    ],
    extra_body={"guided_json": ety_schema},
)
# this crashes vllm server because of the recursive schema. should work once new
# vllm version is released that supports xgrammar 0.1.14
# https://github.com/vllm-project/vllm/issues/13986
print(completion.choices[0].message.content)

In [ ]:
import xgrammar as xgr
import torch
import numpy as np
from transformers import AutoTokenizer, AutoConfig

# Get tokenizer info
tokenizer = AutoTokenizer.from_pretrained(model_name)
config = AutoConfig.from_pretrained(model_name)
# This can be larger than tokenizer.vocab_size due to paddings
full_vocab_size = config.vocab_size
tokenizer_info = xgr.TokenizerInfo.from_huggingface(
    tokenizer, vocab_size=full_vocab_size
)

compiler = xgr.GrammarCompiler(tokenizer_info, max_threads=8)

In [ ]:
# compiling Etymology crashes -- should work after upgrading xgrammar from 0.1.11 to 0.1.14
# compiled_grammar = compiler.compile_json_schema(Etymology)
# compiled_grammar = compiler.compile_json_schema(Item)